# TDDFT vs TDDFT+SOC — group-14 atoms (C, Si, Ge)

Compare **bare closed-shell TDA** (singlets + triplets) with **one-electron SI-SOC** mixing for carbon, silicon, and germanium.

## What this shows

- Singlet and triplet excitation ladders from RKS-TDA
- State-interaction SOC (`casidapy.utils.soc`) using PySCF `int1e_ia01p`
- SOC **grows down the group** (C → Si → Ge): larger S–T mixing and borrowed oscillator strength

## Reference choice

Neutral C/Si/Ge have open-shell \(^3P\) ground terms. This notebook uses the **closed-shell dications** \(\mathrm{C}^{2+}\), \(\mathrm{Si}^{2+}\), \(\mathrm{Ge}^{2+}\) (`charge=2`, `spin=0`) so the CasidaPy singlet↔triplet SI-SOC path applies on a stable RKS reference. The Z-dependent SOC trend is the point of the demo.

Defaults: `sto-3g` + `pbe`, small root counts (login-node friendly).


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, dft

from casidapy import extract_gto_kernel, run_casida, solve_soc_si

HA_TO_EV = 27.211386245988
XC = "pbe"
BASIS = "sto-3g"
N_S = 4
N_T = 4
ATOMS = ["C", "Si", "Ge"]  # run as M^{2+} closed-shell

print(f"defaults: XC={XC}, basis={BASIS}, n_S={N_S}, n_T={N_T}")


In [ ]:
def run_atom(symbol, xc=XC, basis=BASIS, n_s=N_S, n_t=N_T):
    """Closed-shell RKS + singlet/triplet TDA + SI-SOC for one atom."""
    from casidapy.utils.soc import st_soc_matrix, soc_ao_integrals, soc_mo_blocks

    mol = gto.M(atom=f"{symbol} 0 0 0", basis=basis, spin=0, charge=2, verbose=0)
    mf = dft.RKS(mol)
    mf.xc = xc
    mf.grids.level = 1
    e_scf = mf.kernel()
    if not mf.converged:
        print(f"  warning: SCF not converged for {symbol}")

    ks, opts_s = extract_gto_kernel(
        mf, n_states=n_s, tda=True, use_df=False, spin_state="singlet",
    )
    kt, opts_t = extract_gto_kernel(
        mf, n_states=n_t, tda=True, use_df=False, spin_state="triplet",
    )
    opts_s.solver_method = opts_t.solver_method = "eigsh"
    res_s = run_casida(ks, opts_s)
    res_t = run_casida(kt, opts_t)
    soc = solve_soc_si(res_s, res_t, ks, include_ground=False)

    # Characteristic SOC scale: Frobenius norm of S–T coupling blocks
    hso = soc_ao_integrals(mol)
    h_oo, h_vv, _ = soc_mo_blocks(hso, ks._C_o, ks._C_v)
    n_o, n_v = ks.n_occ, ks.n_unocc
    Xs = res_s.Z.T.reshape(res_s.Z.shape[1], n_o, n_v)
    Xt = res_t.Z.T.reshape(res_t.Z.shape[1], n_o, n_v)
    H_st = st_soc_matrix(Xs, Xt, h_oo, h_vv)
    soc_scale = float(np.linalg.norm(H_st))

    return {
        "symbol": symbol,
        "Z": mol.atom_charge(0),
        "E_scf": e_scf,
        "omega_s": res_s.omega.copy(),
        "f_s": res_s.f.copy(),
        "omega_t": res_t.omega.copy(),
        "omega_soc": soc.omega.copy(),
        "f_soc": soc.f.copy(),
        "singlet_w": soc.singlet_weight.copy(),
        "triplet_w": soc.triplet_weight.copy(),
        "soc_scale": soc_scale,
    }


def print_atom(data, nprint=6):
    sym = data["symbol"]
    print(f"\n=== {sym}^{{2+}} (Z={data['Z']})  E_SCF={data['E_scf']:.6f} Ha ===")
    print(f"  ||H_ST||_F = {data['soc_scale']*HA_TO_EV:.4f} eV")
    print("  Singlet TDA:")
    for i, w in enumerate(data["omega_s"][:nprint]):
        print(f"    S{i}: {w*HA_TO_EV:8.3f} eV   f={data['f_s'][i]:.4e}")
    print("  Triplet TDA:")
    for i, w in enumerate(data["omega_t"][:nprint]):
        print(f"    T{i}: {w*HA_TO_EV:8.3f} eV")
    print("  SI-SOC mixed (lowest):")
    for i, w in enumerate(data["omega_soc"][:nprint]):
        print(
            f"    {i}: {w*HA_TO_EV:8.3f} eV   "
            f"S={data['singlet_w'][i]:.3f} T={data['triplet_w'][i]:.3f}  "
            f"f={data['f_soc'][i]:.4e}"
        )


In [ ]:
results = []
for sym in ATOMS:
    print(f"running {sym} ...")
    data = run_atom(sym)
    print_atom(data)
    results.append(data)


In [ ]:
# --- Comparison plots ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4.0))

# 1) SOC coupling scale vs Z
Zs = [d["Z"] for d in results]
scales = [d["soc_scale"] * HA_TO_EV for d in results]
axes[0].plot(Zs, scales, "o-", ms=8)
for d, s in zip(results, scales):
    axes[0].annotate(d["symbol"], (d["Z"], s), textcoords="offset points", xytext=(6, 4))
axes[0].set_xlabel("atomic number Z")
axes[0].set_ylabel(r"$||H_{ST}||_F$ (eV)")
axes[0].set_title("SOC coupling scale")

# 2) Stick spectra: bare singlets vs SOC (Ge as example of strongest SOC)
d = results[-1]
ax = axes[1]
ax.vlines(d["omega_s"] * HA_TO_EV, 0, np.maximum(d["f_s"], 1e-6), colors="C0", lw=1.6, label="singlet TDA")
ax.vlines(d["omega_t"] * HA_TO_EV, 0, 0.05, colors="C1", lw=1.2, alpha=0.7, label="triplet TDA")
ax.set_title(f"{d['symbol']}: bare TDA")
ax.set_xlabel("ω (eV)")
ax.set_ylabel("f (singlet) / arb (triplet)")
ax.legend(fontsize=8)

ax = axes[2]
fplot = np.maximum(d["f_soc"], 1e-8)
sc = ax.scatter(d["omega_soc"] * HA_TO_EV, fplot, c=d["triplet_w"], cmap="coolwarm",
                vmin=0, vmax=1, s=50, zorder=3)
ax.vlines(d["omega_soc"] * HA_TO_EV, 0, fplot, colors="0.7", lw=1.0, zorder=1)
ax.set_title(f"{d['symbol']}: TDA + SI-SOC")
ax.set_xlabel("ω (eV)")
ax.set_ylabel("f (borrowed)")
fig.colorbar(sc, ax=ax, label="triplet weight")

fig.suptitle(f"Group-14 dications M$^{{2+}}$ — {XC}/{BASIS}", y=1.02)
fig.tight_layout()
plt.show()

print("Summary:")
for d in results:
    # max triplet weight among states with f > small threshold
    bright = d["f_soc"] > 1e-6
    tmax = float(np.max(d["triplet_w"][bright])) if np.any(bright) else float(np.max(d["triplet_w"]))
    print(f"  {d['symbol']:2s}: ||H_ST||={d['soc_scale']*HA_TO_EV:.4f} eV, "
          f"max triplet wt (bright-ish)={tmax:.3f}")


## Notes

- Integral: one-electron BP-like SOC only (`int1e_ia01p` × \(1/(2c^2)\)); no 2e screening.
- SI basis: singlets ⊕ Cartesian triplet components \((T_x,T_y,T_z)\); no triplet–triplet SOC yet.
- For a stronger SOC demo with the same machinery, try `scripts/run_soc_qed_demo.py --molecule hi`.
